In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_models, get_features, ModelTypes
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

flash attention installed


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dv2_tr', 'dv2_cb', 'dv2_db', 'dvt')
# enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dv2_tr',)
# enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dv2_b', 'dv2_cb',  'dvt', 'alibi_dv2_coco')
# enabled_models: tuple[ModelTypes, ...] = ('dv2',)

models = get_models(enabled_models, '../../trained_models', device=DEVICE, conf_path='../../dinov3')

debiased?


In [3]:
# ds_folder = '../paper_figures/data/linear_probe/homog_micros'
ds_folder = '../paper_figures/data/linear_probe/texture_ds'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

replace_with_random_noise: bool = True

features = {k: [] for k in models.keys()}
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')

    for i, (model_name, model) in enumerate(models.items()):

        channel_blank = 'cb' in model_name
        avg_over_trs = 'tr' in model_name
        feats = get_features(model, img, device=DEVICE, channel_blank=channel_blank, channel_last=True, avg_over_trs=avg_over_trs)
        features[model_name].append(feats)

In [4]:
# import matplotlib.pyplot as plt
# from dinosaw.utils import do_2D_pca

# n_imgs = 4
# fig, axs = plt.subplots(2, n_imgs, figsize=(n_imgs*3, 6))


# for j, model in enumerate(features.keys()):
#     for i in range(n_imgs):
#         pcaed = do_2D_pca(features[model][4 + i].transpose((2, 0, 1)), post_norm='minmax')
#         axs[j, i].imshow(pcaed)


In [5]:
ramp: RampTypes = 'radial'
ramps_to_results: dict[ModelTypes, list[LinearProbeResult]] = {m: [] for m in enabled_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True
N_repeats = 6

for model_name in enabled_models:
    for i in range(n_imgs):
        feats = features[model_name][i]
        for i in range(N_repeats):
            result = do_linear_probe(feats, ramp, probe_by_channel=False, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK, regressor='ridge')
            ramps_to_results[model_name].append(result)

In [6]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)


    if results[0]['per_channel_scores'] is None:
        return None, None, mean_score, std_score, mean_pred

    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [7]:
for model_name in enabled_models:
    results = average_results(ramps_to_results[model_name])
    print(f"{model_name}: {results[2]:.2f} ± {results[3]:.2f}")

dv2: 0.87 ± 0.02
dv2_tr: 0.93 ± 0.01
dv2_cb: 0.87 ± 0.02
dv2_db: 0.86 ± 0.03
dvt: 0.92 ± 0.02


In [8]:
# lr, micro
# dv2: 0.79 ± 0.07
# dv2_tr: 0.65 ± 0.14
# dv2_cb: 0.71 ± 0.11
# dv2_db: 0.63 ± 0.13
# dvt: 0.53 ± 0.22

In [9]:
# lr texture
# dv2: 0.68 ± 0.21
# dv2_tr: 0.54 ± 0.30
# dv2_cb: 0.62 ± 0.27
# dv2_db: 0.52 ± 0.28
# dvt: 0.45 ± 0.42

In [10]:
# lr, noise
# dv2: 0.86 ± 0.03
# dv2_tr: 0.67 ± 0.08
# dv2_cb: 0.85 ± 0.03
# dv2_db: 0.80 ± 0.05
# dvt: 0.69 ± 0.07

In [11]:
# ud, micro
# dv2: 0.86 ± 0.06
# dv2_tr: 0.59 ± 0.16
# dv2_cb: 0.84 ± 0.07
# dv2_db: 0.84 ± 0.07
# dvt: 0.95 ± 0.02

In [12]:
# ud, texture
# dv2: 0.74 ± 0.21
# dv2_tr: 0.53 ± 0.36
# dv2_cb: 0.73 ± 0.21
# dv2_db: 0.78 ± 0.18
# dvt: 0.91 ± 0.18

In [13]:
# ud, noise
# dv2: 0.93 ± 0.02
# dv2_tr: 0.79 ± 0.05
# dv2_cb: 0.92 ± 0.02
# dv2_db: 0.92 ± 0.02
# dvt: 0.97 ± 0.01

In [14]:
# radial, micro
# dv2: 0.81 ± 0.07
# dv2_tr: 0.83 ± 0.07
# dv2_cb: 0.78 ± 0.08
# dv2_db: 0.79 ± 0.09
# dvt: 0.84 ± 0.07

In [15]:
# radial, texture
# dv2: 0.70 ± 0.19
# dv2_tr: 0.73 ± 0.18
# dv2_cb: 0.67 ± 0.22
# dv2_db: 0.66 ± 0.22
# dvt: 0.73 ± 0.32

In [ ]:
# radial, noise
# dv2: 0.87 ± 0.02
# dv2_tr: 0.93 ± 0.01
# dv2_cb: 0.87 ± 0.02
# dv2_db: 0.86 ± 0.03
# dvt: 0.92 ± 0.02